In [ ]:
def print_2d_matrix(lst, raw=False):
    print(*[num for num in lst], sep="\n")
    return
    print(*[round(num, 8) for num in lst], sep="\n")

In [ ]:
import numpy as np

q = ["cold", "hot"]
N = len(q)

# a[i][j] probability of moving from state i to state j
a = np.random.dirichlet(np.ones(N), size=N)

# All of the observations available
v = ["1", "2", "3"]

# b[i][j] probability of getting observation j knowing state i happened
b = [
    dict(zip(v, np.random.dirichlet(np.ones(len(v)))))
    for _ in range(N)
]

pi = np.squeeze(np.random.dirichlet(np.ones(N), size=1))

# print_2d_matrix(a)
# print_2d_matrix(b, raw=True)
# print_2d_matrix(pi)
print(a)
print()
print(b)
print()
print(pi)

[[0.04081988 0.95918012]
 [0.96053509 0.03946491]]

[{'1': np.float64(0.1544775618965409), '2': np.float64(0.2268643003184488), '3': np.float64(0.6186581377850102)}, {'1': np.float64(0.1370419688170287), '2': np.float64(0.6425753407089347), '3': np.float64(0.22038269047403658)}]

[0.36269498 0.63730502]


In [ ]:
o = ["2", "1", "1", "3"]
T = len(o)

alpha = np.zeros((N, T), dtype=float)
beta = np.zeros((N, T), dtype=float)

# gamma = [[0 for _ in range(T)] for _ in range(N)]


In [ ]:
def update_alpha():
    global alpha
    alpha = np.zeros((N, T), dtype=float)
    for state in range(N):
        alpha[state, 0] = pi[state] * b[state][o[0]]

    for t in range(1, T):
        observation = o[t]
        for state in range(N):
            for state_prev in range(N):
                alpha[state][t] += alpha[state_prev][t - 1] * a[state_prev][state] * b[state][observation]

update_alpha()
print(alpha)

[[0.08228254 0.06128337 0.00231994 0.00488743]
 [0.40951649 0.01303068 0.00812605 0.00056108]]


In [ ]:
def update_beta():
    global beta
    beta = np.zeros((N, T), dtype=float)
    beta[:, -1] = 1.0

    for t in range(T-2, -1, -1):
        observation = o[t + 1]
        for state in range(N):
            for state_next in range(N):
                beta[state][t] += beta[state_next][t + 1] * a[state][state_next] * b[state_next][observation]

update_beta()
print(beta)

[[0.00555334 0.08074744 0.23664025 1.        ]
 [0.01218894 0.03837386 0.60294023 1.        ]]


In [ ]:
# The probability of being in state i at time t and j at time t+1
def xi(t, i, j):
    not_quite_xi = alpha[i, t] * a[i, j] * b[j][o[t + 1]] * beta[j, t+1]
    denominator = np.sum(alpha[:, t] * beta[:, t])
    return not_quite_xi / denominator

In [ ]:
# The probability of being in state j at time t
def gamma(t, j):
    numerator = alpha[j, t] * beta[j, t]
    denominator = np.sum(alpha[:, t] * beta[:, t])
    return numerator / denominator

In [ ]:
def update_a():
    global a

    new_a = np.zeros_like(a)

    for i in range(N):
        denominator = sum(gamma(t, i) for t in range(T-1))

        for j in range(N):
            numerator = sum(xi(t, i, j) for t in range(T-1))
            new_a[i, j] = numerator / denominator

    a = new_a

In [ ]:
def update_b():
    global b

    new_b = [dict() for _ in range(N)]

    for j in range(N):
        denominator = sum(gamma(t, j) for t in range(T))

        for symbol in v:
            numerator = sum(
                gamma(t, j)
                for t in range(T)
                if o[t] == symbol
            )

            new_b[j][symbol] = numerator / denominator

    b = new_b

In [ ]:
def update_pi():
    global pi

    for i in range(N):
        pi[i] = gamma(0, i)

In [ ]:
i = 1
prev_log_likelihood = -np.inf
while True:

    update_alpha()
    log_likelihood = ...

    update_beta()

    update_a()
    update_b()
    update_pi()

    if abs(log_likelihood - prev_log_likelihood) < 1e-6:
        break

    prev_log_likelihood = log_likelihood

1 -5.212412612632896
1 -3.115749203685783
1 -2.7979664341232824
1 -2.772996288083958
1 -2.772588888606059
1 -2.772588722239817
